## Mostly close code in python

In [3]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import math
import numpy as np
import datetime
import matplotlib
matplotlib.use("TkAgg")
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

# =====================================================================
# 1. GEOMETRY & FELLENIUS ENGINE
# =====================================================================

def create_slope_geometry(H_cut, cut_angle_deg, H_nat, nat_angle_deg):
    """ Creates the x, y coordinates starting from 0,0 based on heights and angles. """
    x_toe_start = 0.0
    x_toe_end = 20.0
    p1 = (x_toe_start, 0.0)
    p2 = (x_toe_end, 0.0)
    
    if cut_angle_deg <= 0 or cut_angle_deg >= 90:
        raise ValueError("Cut slope angle must be between 0 and 90 degrees.")
        
    dx_cut = H_cut / np.tan(np.radians(cut_angle_deg))
    x_crest = x_toe_end + dx_cut
    p3 = (x_crest, H_cut)
    
    if H_nat > 0 and nat_angle_deg > 0:
        if nat_angle_deg >= 90:
            raise ValueError("Natural slope angle must be less than 90 degrees.")
        dx_nat = H_nat / np.tan(np.radians(nat_angle_deg))
        x_top = x_crest + dx_nat
        y_top = H_cut + H_nat
    else:
        x_top = x_crest + 10.0
        y_top = H_cut
    p4 = (x_top, y_top)
    
    # Returning exactly up to the crest of the natural slope
    return [p1, p2, p3, p4]

class SlopeStabilityAnalyzer:
    def __init__(self, terrain_points, c, phi_deg, gamma, ru):
        self.points = sorted(terrain_points, key=lambda p: p[0])
        self.xs = np.array([p[0] for p in self.points])
        self.ys = np.array([p[1] for p in self.points])
        
        self.H_total = np.max(self.ys) - np.min(self.ys)
        self.W_total = np.max(self.xs) - np.min(self.xs)
            
        self.c = c
        self.phi = np.radians(phi_deg)
        self.gamma = gamma
        self.ru = ru
        
        self.x_toe = self.xs[1]
        self.x_crest = self.xs[2]

    def generate_auto_grid(self, resolution):
        xc_range = (self.x_toe, self.xs[-1])
        yc_range = (np.max(self.ys) + 5, np.max(self.ys) + 2.5 * self.H_total)
        R_range = (0.5 * self.H_total, 3.0 * self.H_total)
        step = max(self.H_total, self.W_total) / resolution
        return xc_range, yc_range, R_range, step

    def find_entry_exit(self, xc, yc, R):
        if yc - R > np.min(self.ys): return None, None
        x_scan = np.linspace(self.xs[0], self.xs[-1], 500)
        gy = np.interp(x_scan, self.xs, self.ys)
        rad_term = R**2 - (x_scan - xc)**2
        valid = rad_term >= 0
        if not np.any(valid): return None, None
        cy = np.full_like(x_scan, np.nan)
        cy[valid] = yc - np.sqrt(rad_term[valid])
        diff = gy - cy
        inside = diff > 0
        crossings = np.where(np.diff(inside))[0]
        if len(crossings) < 2: return None, None 
        def exact_intersect(idx):
            x_a, x_b = x_scan[idx], x_scan[idx+1]
            d_a, d_b = diff[idx], diff[idx+1]
            return x_a - d_a * (x_b - x_a) / (d_b - d_a + 1e-12)
        return exact_intersect(crossings[0]), exact_intersect(crossings[-1])

    def calculate_fos(self, xc, yc, R, limit_entry_exit=True, n_slices=30):
        x_start, x_end = self.find_entry_exit(xc, yc, R)
        if x_start is None: return float('inf')

        if limit_entry_exit:
            if x_start > self.x_toe + 5: return float('inf') 
            if x_end < self.x_crest: return float('inf')

        x_check = np.linspace(x_start, x_end, 10)
        y_check = np.interp(x_check, self.xs, self.ys)
        if np.max(y_check) - np.min(y_check) < 0.05 * self.H_total: return float('inf')

        dx = (x_end - x_start) / n_slices
        resisting_sum = driving_sum = 0.0
        slices_data = []

        for i in range(n_slices):
            xm = x_start + i * dx + (dx / 2)
            sin_alpha = np.clip((xm - xc) / R, -0.999, 0.999)
            alpha = np.arcsin(sin_alpha)
            
            y_ground = float(np.interp(xm, self.xs, self.ys))
            y_circle = yc - R * np.cos(alpha)
            h = y_ground - y_circle
            if h <= 0: continue
            
            A = h * dx
            W = self.gamma * A
            L = dx / np.cos(alpha)
            u = self.ru * self.gamma * h       
            N = W * np.cos(alpha)              
            N_prime = max(N - u * L, 0)        
            T = W * np.sin(alpha)              
            S = self.c * L + N_prime * np.tan(self.phi) 
            
            resisting_sum += S
            driving_sum += T
            slices_data.append({'x': xm, 'y': y_circle, 'h': h, 'alpha': alpha})

        if driving_sum <= 0: return float('inf')
        return resisting_sum / driving_sum, slices_data, x_start, x_end

    def analyze(self, resolution=30, n_slices=30, limit_entry_exit=True):
        xc_range, yc_range, R_range, step = self.generate_auto_grid(resolution)
        best_fos, best_circle, best_data = float('inf'), None, None
        
        for xc in np.arange(*xc_range, step):
            for yc in np.arange(*yc_range, step):
                for R in np.arange(*R_range, step):
                    result = self.calculate_fos(xc, yc, R, limit_entry_exit, n_slices)
                    if isinstance(result, tuple):
                        fos, slices, x_entry, x_exit = result
                        if fos < best_fos:
                            best_fos, best_circle, best_data = fos, (xc, yc, R), (slices, x_entry, x_exit)
        return best_fos, best_circle, best_data

# =====================================================================
# 2. LOOKUP DATABASES & CONSTANTS
# =====================================================================

SOIL_DB = {
    'CG': {'name': 'Coarse-grained (Clean) — GW, SW, GP, SP', 'I': {'c': 0.0, 'phi': 34.0, 'g': 19.0}, 'L': {'c': 0.0, 'phi': 30.0, 'g': 18.0}, 'H': {'c': 2.0, 'phi': 38.0, 'g': 20.0}, 'hint': 'Coarse-grained (Clean): GW, SW, GP, SP. Well drained, cohesionless. Poses high erosion risk on slopes.'},
    'CGNPF': {'name': 'Coarse-grained + non-plastic fines — SM, GM', 'I': {'c': 5.0, 'phi': 31.0, 'g': 18.5}, 'L': {'c': 2.0, 'phi': 28.0, 'g': 18.0}, 'H': {'c': 8.0, 'phi': 34.0, 'g': 19.5}, 'hint': 'Coarse-grained + non-plastic: SM, GM. Silty sands/gravels. Highly susceptible to rain-induced gullying.'},
    'CGPF': {'name': 'Coarse-grained + plastic fines — GC, SC', 'I': {'c': 12.0, 'phi': 28.0, 'g': 19.0}, 'L': {'c': 8.0, 'phi': 25.0, 'g': 18.5}, 'H': {'c': 16.0, 'phi': 31.0, 'g': 20.0}, 'hint': 'Coarse-grained + plastic: GC, SC. Clayey gravels and sands. Prone to shallow creeping slumps.'},
    'ML': {'name': 'Silt / Micaceous Silt (ML / MH)', 'I': {'c': 10.0, 'phi': 26.0, 'g': 17.5}, 'L': {'c': 5.0, 'phi': 22.0, 'g': 17.0}, 'H': {'c': 15.0, 'phi': 29.0, 'g': 18.0}, 'hint': 'Silt: ML, MH. Common in mid-hills fill slopes. Extremely low wet-strength.'},
    'CL': {'name': 'Low plasticity clay (CL)', 'I': {'c': 18.0, 'phi': 22.0, 'g': 18.0}, 'L': {'c': 10.0, 'phi': 18.0, 'g': 17.5}, 'H': {'c': 25.0, 'phi': 25.0, 'g': 19.0}, 'hint': 'Low plasticity clay: CL. Sandy/silty clays. Retains structural form when un-saturated.'},
    'CH': {'name': 'High plasticity clay (CH)', 'I': {'c': 25.0, 'phi': 15.0, 'g': 17.0}, 'L': {'c': 15.0, 'phi': 10.0, 'g': 16.5}, 'H': {'c': 35.0, 'phi': 18.0, 'g': 18.0}, 'hint': 'High plasticity clay: CH. Fat clays / expansive soils. Prone to severe shrink-swell deformation.'}
}

RUNOFF_TYPES = [
    ("Steep bare rock; concrete / bitumen pavement surface", 0.90),
    ("Steep rock with some vegetative cover", 0.80),
    ("Bare stiff clay (impervious soil); cut slope", 0.70),
    ("Stiff clay with vegetative cover; compacted gravel road", 0.60),
    ("Loam soil, lightly cultivated or covered; disturbed slope", 0.50),
    ("Loam largely cultivated, turfed, or grassed", 0.40),
    ("Sandy soil, light growth, parks, meadows", 0.30),
    ("Sandy soil with heavy brush or forested areas", 0.20)
]

LINING_DB = {
    "Smooth trowel-finished concrete": {"n_disp": "0.012–0.014", "n": 0.013, "v_disp": "6.0", "v": 6.0},
    "Float-finished concrete": {"n_disp": "0.013–0.015", "n": 0.014, "v_disp": "6.0", "v": 6.0},
    "Cement stone masonry (1:4), random rubble": {"n_disp": "0.017–0.020", "n": 0.0185, "v_disp": "4.5–5.0", "v": 4.75},
    "Dressed stone in mortar": {"n_disp": "0.015–0.017", "n": 0.016, "v_disp": "5.0–6.0", "v": 5.5},
    "Rough stone / riprap lining": {"n_disp": "0.025–0.030", "n": 0.0275, "v_disp": "4.0–4.5", "v": 4.25},
    "Gabion lining": {"n_disp": "0.028–0.035", "n": 0.0315, "v_disp": "3.5–4.5", "v": 4.0},
    "Dense turf (grass-lined)": {"n_disp": "0.030–0.050", "n": 0.040, "v_disp": "0.9–1.2", "v": 1.05},
    "Bare earth, no protection": {"n_disp": "0.020–0.026", "n": 0.023, "v_disp": "0.3–0.6", "v": 0.45},
    "Bituminous / asphalt surface": {"n_disp": "0.013–0.016", "n": 0.0145, "v_disp": "5.0–6.0", "v": 5.5},
    "Natural rock channel (smooth and uniform)": {"n_disp": "0.035–0.040", "n": 0.0375, "v_disp": "5.5–6.0", "v": 5.75},
    "Natural rock channel (Jagged)": {"n_disp": "0.040–0.045", "n": 0.0425, "v_disp": "2.5", "v": 2.5},
    "Natural rock channel (Soft)": {"n_disp": "0.035", "n": 0.035, "v_disp": "2.5", "v": 2.5},
    "Natural rock channel (Hard)": {"n_disp": "0.035", "n": 0.035, "v_disp": "5.5", "v": 5.5}
}

# =====================================================================
# 3. SECONDARY ENGINEERING CALCULATIONS
# =====================================================================

def calc_rankine_retaining_wall(Hw, B, a, Df, gamma_wall, mu, phi_soil, gamma_soil):
    phi_rad = math.radians(phi_soil)
    ka = (1 - math.sin(phi_rad)) / (1 + math.sin(phi_rad))
    H_total = Hw + Df
    Pa = 0.5 * ka * gamma_soil * (H_total ** 2)
    Mo = Pa * (H_total / 3.0)               
    w_stem_rectangular = a * Hw * gamma_wall
    w_stem_triangular = 0.5 * (B - a) * Hw * gamma_wall
    w_foundation = B * Df * gamma_wall
    W_total = w_stem_rectangular + w_stem_triangular + w_foundation
    if W_total <= 0: W_total = 0.001
    Mr = (w_stem_rectangular * (B - (a / 2.0))) + (w_stem_triangular * ((B - a) * (2.0 / 3.0))) + (w_foundation * (B / 2.0))
    fos_overturning = Mr / Mo if Mo > 0 else 9.9
    fos_sliding = (W_total * mu) / Pa if Pa > 0 else 9.9
    x_centroid = (Mr - Mo) / W_total
    ecc = (B / 2.0) - x_centroid
    q_max = (W_total / B) * (1 + 6 * abs(ecc) / B) if B > 0 else 0
    return fos_overturning, fos_sliding, ecc, q_max

def calc_kirpich_tc(L, H):
    if H > 0 and L > 0:
        tc_raw = 0.0195 * (L ** 1.155) / (H ** 0.385)
        tc_used = max(5.0, min(20.0, tc_raw))
        return tc_raw, tc_used
    return 0.0, 0.0

def calc_rational_discharge(C, I_mm_hr, A_km2):
    A_ha = A_km2 * 100 
    Qpeak = (C * I_mm_hr * A_ha) / 360.0 
    return Qpeak

def calc_manning_hydraulics(profile_type, B, D, z, S_pct, n):
    S = S_pct / 100.0
    if profile_type == "Rectangular":
        z = 0.0 
    Area_flow = (B * D) + (z * (D ** 2))
    Wet_perim = B + (2 * D * math.sqrt(1 + (z ** 2)))
    R = Area_flow / Wet_perim if Wet_perim > 0 else 0.0
    V = (1.0 / n) * (R ** (2.0 / 3.0)) * (S ** 0.5)
    Qcap = Area_flow * V
    return Qcap, V

# =====================================================================
# 4. GUI AND RENDERING ENGINE (TKINTER)
# =====================================================================

C_EARTH = "#2C3E2D"; C_EARTH_L = "#3D5C3F"
C_SAGE = "#7A9E7E"; C_SAGE_LL = "#D4E8D6"
C_CLAY = "#C0603A"; C_AMBER = "#E8A03C"
C_BG = "#F5F3EE"; C_BG2 = "#ECEAE4"
C_TEXT = "#1A1F1A"; C_TEXT_LIGHT = "#7A8C7C"
C_BORDER = "#D0CEC8"
C_SUCCESS = "#2D7A4F"; C_SUCCESS_L = "#E8F4EC"
C_INFO = "#2A5F8A"; C_INFO_L = "#E8F0F8"

class RoadsideStabilizationApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Roadside Slope Stabilization — Bagmati Province Field Tool")
        self.geometry("1150x900")
        self.configure(bg=C_BG)

        self.state = {
            "alt": 1400, "aspect": "North (N)", "land": "Forest", "moisture": "moist - damp, some seepage", "rain": "Medium",
            "isRockMass": False, "soilType": "CG", "strClass": "I", "beta": 10.0, "alpha": 40.0, "H": 6.0, "H_nat": 10.0, "ru": 0.25, 
            "res": 30, "slices": 30,
            "rockGroup": "G1", "rockMass": "blocky", "roughness": "rough", "rH": 8.0, "bdip": 35.0, "bdir": "into", "sfa": 60.0,
            "fsCut": 1.2, "wallNeeded": True, 
            "wH": 3.0, "wB": 1.8, "wa": 0.5, "wDf": 1.0, "wSbcVal": 150.0,
            "dArea_km2": 0.05, "dtProfile": "Trapezoidal", "dtB": 0.4, "dtz": 1.0, "dtD": 0.5, "dtS": 2.0, 
            "tc_L": 300.0, "tc_H": 45.0, "tc_raw": 0.0, "tc_used": 0.0, "dI": 80.0
        }
        self.current_cw = 0.0 
        self.current_step = 1

        self.setup_styles()
        self.build_ui()
        self.show_page(1)

    def setup_styles(self):
        style = ttk.Style(); style.theme_use('clam')
        style.configure(".", background=C_BG, foreground=C_TEXT, font=("Segoe UI", 10))
        style.configure("Header.TFrame", background=C_EARTH); style.configure("Nav.TFrame", background=C_EARTH_L)
        style.configure("Action.TButton", font=("Segoe UI Semibold", 10), background=C_EARTH, foreground="white", borderwidth=0)
        style.map("Action.TButton", background=[("active", C_EARTH_L)])
        style.configure("Nav.TButton", font=("Segoe UI Semibold", 9), background=C_BG2, foreground=C_TEXT, borderwidth=1)

    def build_ui(self):
        header = ttk.Frame(self, style="Header.TFrame"); header.pack(fill="x", side="top")
        tk.Label(header, text="Roadside Slope Stabilization", font=("Georgia", 16, "bold"), fg="white", bg=C_EARTH).pack(anchor="w", padx=20, pady=(10, 2))
        tk.Label(header, text="FIELD ASSESSMENT & DESIGN TOOL  ·  BAGMATI PROVINCE, NEPAL", font=("Segoe UI", 8, "bold"), fg="#A8C5AB", bg=C_EARTH).pack(anchor="w", padx=20, pady=(0, 10))

        self.nav_frame = ttk.Frame(self, style="Nav.TFrame"); self.nav_frame.pack(fill="x")
        self.nav_buttons = []
        steps_info = [("1. Site Material", 1), ("2. Stability Analysis", 2), ("3. Retaining Wall", 3), ("4. Bioengineering", 4), ("5. Drain Design", 5)]
        for name, num in steps_info:
            btn = tk.Button(self.nav_frame, text=name, bg=C_EARTH_L, fg="white", activebackground=C_EARTH, activeforeground="white", font=("Segoe UI Semibold", 9), bd=0, relief="flat", padx=15, pady=8, command=lambda n=num: self.show_page(n))
            btn.pack(side="left", fill="y"); self.nav_buttons.append(btn)

        self.container = tk.Frame(self, bg=C_BG); self.container.pack(fill="both", expand=True)
        self.canvas = tk.Canvas(self.container, borderwidth=0, highlightthickness=0, bg=C_BG)
        self.scrollbar = ttk.Scrollbar(self.container, orient="vertical", command=self.canvas.yview)
        self.scrollable_frame = tk.Frame(self.canvas, bg=C_BG)
        self.scrollable_frame.bind("<Configure>", lambda e: self.canvas.configure(scrollregion=self.canvas.bbox("all")))
        self.canvas.bind("<Configure>", lambda e: self.canvas.itemconfig(self.canvas_window, width=e.width))
        self.canvas.bind_all("<MouseWheel>", lambda e: self.canvas.yview_scroll(int(-1*(e.delta/120)), "units"))
        self.canvas_window = self.canvas.create_window((0, 0), window=self.scrollable_frame, anchor="nw")
        self.canvas.configure(yscrollcommand=self.scrollbar.set)
        self.canvas.pack(side="left", fill="both", expand=True); self.scrollbar.pack(side="right", fill="y")

        self.pages = {}
        self.build_page_1()
        self.build_page_2()
        self.build_page_3()
        self.build_page_4()
        self.build_page_5()

    def show_page(self, page_num):
        if page_num > 1:
            if not self.sync_page_1_data(): return
            try:
                if page_num == 2:
                    self.ax.clear(); self.ax.set_axis_off()
                    self.ax.text(0.5, 0.5, "Click 'Run Mathematical Assessment' to compute slip surface.", ha='center', va='center', size=11, color=C_TEXT_LIGHT)
                    self.chart_canvas.draw()
                    self.lbl_slope_fs.configure(text="Slope FoS: --")
                    self.lbl_decision.configure(text="Please run stability assessment above.", fg=C_TEXT)
                elif page_num == 3: self.run_wall_design()
                elif page_num == 4: self.build_bioengineering_view()
                elif page_num == 5: 
                    self.update_tc()
                    self.update_cw_totals()
            except Exception as e:
                messagebox.showerror("Execution Error", f"An internal error prevented loading the page:\n\n{str(e)}")
                return

        self.current_step = page_num
        for i, btn in enumerate(self.nav_buttons):
            if i + 1 == page_num: btn.configure(bg=C_EARTH, font=("Segoe UI Bold", 9))
            else: btn.configure(bg=C_EARTH_L, font=("Segoe UI Semibold", 9))

        for p_idx, frame in self.pages.items():
            if p_idx == page_num: frame.pack(fill="both", expand=True, padx=20, pady=15)
            else: frame.pack_forget()
        self.canvas.yview_moveto(0)

    # -----------------------------------------------------------------
    # GUI BUILDERS
    # -----------------------------------------------------------------
    def create_card(self, parent, title, subtitle=None):
        card = tk.Frame(parent, bg="white", highlightbackground=C_BORDER, highlightthickness=1, bd=0)
        card.pack(fill="x", pady=8, padx=5)
        inner = tk.Frame(card, bg="white"); inner.pack(fill="both", expand=True, padx=20, pady=15)
        tk.Label(inner, text=title, font=("Segoe UI Semibold", 12), fg=C_EARTH, bg="white").pack(anchor="w", pady=(0, 2))
        if subtitle: tk.Label(inner, text=subtitle, font=("Segoe UI", 9, "italic"), fg=C_TEXT_LIGHT, bg="white").pack(anchor="w", pady=(0, 10))
        ttk.Separator(inner, orient="horizontal").pack(fill="x", pady=(0, 12))
        return inner

    def label_sm(self, parent, text): return tk.Label(parent, text=text, font=("Segoe UI", 8, "bold"), fg=C_TEXT_LIGHT, bg="white")

    def build_page_1(self):
        page = tk.Frame(self.scrollable_frame, bg=C_BG); self.pages[1] = page
        
        card1 = self.create_card(page, "📍 Site Identification", "Basic location and environmental parameters")
        f_grid1 = tk.Frame(card1, bg="white"); f_grid1.pack(fill="x", pady=5)
        self.label_sm(f_grid1, "ALTITUDE (M ASL)").grid(row=0, column=0, sticky="w", padx=5, pady=(5,0))
        self.label_sm(f_grid1, "ASPECT").grid(row=0, column=1, sticky="w", padx=15, pady=(5,0))
        self.label_sm(f_grid1, "LAND USE").grid(row=0, column=2, sticky="w", padx=15, pady=(5,0))
        self.ent_alt = ttk.Entry(f_grid1, width=20); self.ent_alt.insert(0, str(self.state["alt"])); self.ent_alt.grid(row=1, column=0, sticky="w", padx=5, pady=(2, 15))
        self.cb_aspect = ttk.Combobox(f_grid1, values=["North (N)", "North-East (NE)", "East (E)", "South-East (SE)", "South (S)", "South-West (SW)", "West (W)", "North-West (NW)"], state="readonly", width=25); self.cb_aspect.set(self.state["aspect"]); self.cb_aspect.grid(row=1, column=1, sticky="w", padx=15, pady=(2, 15))
        self.cb_land = ttk.Combobox(f_grid1, values=["Forest", "Scrubland / degraded forest", "Cultivated / terraced", "Barren / eroded", "Settlement", "Pasture / grassland"], state="readonly", width=30); self.cb_land.set(self.state["land"]); self.cb_land.grid(row=1, column=2, sticky="w", padx=15, pady=(2, 15))

        self.label_sm(f_grid1, "SITE MOISTURE").grid(row=2, column=0, sticky="w", padx=5, pady=(5,0))
        self.label_sm(f_grid1, "RAINFALL CATEGORY").grid(row=2, column=1, columnspan=2, sticky="w", padx=15, pady=(5,0))
        self.cb_moist = ttk.Combobox(f_grid1, values=["Dry - no visible seepage", "Moist - damp, some seepage", "Wet - active seepage / springs"], state="readonly", width=35); self.cb_moist.set(self.state["moisture"]); self.cb_moist.grid(row=3, column=0, sticky="w", padx=5, pady=(2, 5))
        
        self.rain_var = tk.StringVar(value=self.state["rain"])
        rain_frame = tk.Frame(f_grid1, bg="white"); rain_frame.grid(row=3, column=1, columnspan=2, sticky="w", padx=15, pady=(2, 5))
        tk.Radiobutton(rain_frame, text="Low <1500 mm", variable=self.rain_var, value="Low", bg="white", font=("Segoe UI", 9)).pack(side="left", padx=(0,10))
        tk.Radiobutton(rain_frame, text="Medium 1500-2500 mm", variable=self.rain_var, value="Medium", bg="white", font=("Segoe UI", 9)).pack(side="left", padx=10)
        tk.Radiobutton(rain_frame, text="High >2500 mm", variable=self.rain_var, value="High", bg="white", font=("Segoe UI", 9)).pack(side="left", padx=10)

        self.card2 = self.create_card(page, "🪨 Lithology & Slope Material", "")
        self.label_sm(self.card2, "PRIMARY SLOPE PROBLEM MATERIAL").pack(anchor="w", pady=(0, 5))
        btn_frame = tk.Frame(self.card2, bg="white"); btn_frame.pack(anchor="w", pady=(0, 15))
        self.btn_soil_mode = tk.Button(btn_frame, text="🟤 Soil slope", bg=C_SAGE_LL, fg=C_EARTH, font=("Segoe UI Semibold", 9), relief="solid", bd=1, padx=15, pady=5, command=lambda: self.toggle_material("soil")); self.btn_soil_mode.pack(side="left", padx=(0, 5))
        self.btn_rock_mode = tk.Button(btn_frame, text="⛰ Rock slope", bg="white", fg=C_TEXT_LIGHT, font=("Segoe UI Semibold", 9), relief="solid", bd=1, padx=15, pady=5, command=lambda: self.toggle_material("rock")); self.btn_rock_mode.pack(side="left", padx=5)

        # Soil Panel
        self.soil_panel = tk.Frame(self.card2, bg="#F7F6F2", highlightbackground=C_BORDER, highlightthickness=1, bd=0, padx=15, pady=15)
        self.soil_panel.pack(fill="x")
        self.label_sm(self.soil_panel, "SOIL TYPE (USCS)").grid(row=0, column=0, sticky="w")
        self.label_sm(self.soil_panel, "STRENGTH CLASS").grid(row=0, column=1, sticky="w", padx=15)
        self.cb_soil = ttk.Combobox(self.soil_panel, values=[v["name"] for k,v in SOIL_DB.items()], width=45, state="readonly"); self.cb_soil.set(SOIL_DB[self.state["soilType"]]["name"]); self.cb_soil.grid(row=1, column=0, sticky="w", pady=(2, 10))
        self.cb_str = ttk.Combobox(self.soil_panel, values=["I (Intermediate)", "L (Low/Conservative)", "H (High/Optimistic)"], width=30, state="readonly"); self.cb_str.set("I (Intermediate)"); self.cb_str.grid(row=1, column=1, sticky="w", padx=15, pady=(2, 10))
        self.cb_soil.bind("<<ComboboxSelected>>", self.on_soil_updated); self.cb_str.bind("<<ComboboxSelected>>", self.on_soil_updated)

        self.lbl_soil_hint = tk.Label(self.soil_panel, text="", font=("Segoe UI", 9), fg=C_EARTH, bg=C_SUCCESS_L, wraplength=700, justify="left", anchor="w", padx=10, pady=8); self.lbl_soil_hint.grid(row=2, column=0, columnspan=2, sticky="ew", pady=(5, 10))
        
        self.param_box = tk.Frame(self.soil_panel, bg=C_SAGE_LL, highlightbackground=C_EARTH_L, highlightthickness=1, bd=0, padx=15, pady=10); self.param_box.grid(row=3, column=0, columnspan=2, sticky="ew", pady=(5, 15)); self.param_box.columnconfigure((0,1,2), weight=1)
        self.label_sm(self.param_box, "C' (KPA)").grid(row=0, column=0, sticky="w"); self.label_sm(self.param_box, "Φ' (°)").grid(row=0, column=1, sticky="w"); self.label_sm(self.param_box, "Γ (KN/M³)").grid(row=0, column=2, sticky="w")
        self.lbl_c = tk.Label(self.param_box, text="0", font=("Georgia", 16, "bold"), fg=C_EARTH, bg=C_SAGE_LL); self.lbl_c.grid(row=1, column=0, sticky="w")
        self.lbl_phi = tk.Label(self.param_box, text="34", font=("Georgia", 16, "bold"), fg=C_EARTH, bg=C_SAGE_LL); self.lbl_phi.grid(row=1, column=1, sticky="w")
        self.lbl_gamma = tk.Label(self.param_box, text="19", font=("Georgia", 16, "bold"), fg=C_EARTH, bg=C_SAGE_LL); self.lbl_gamma.grid(row=1, column=2, sticky="w")

        self.label_sm(self.soil_panel, "NATURAL SLOPE ANGLE B (°)").grid(row=4, column=0, sticky="w", pady=(5,0))
        self.label_sm(self.soil_panel, "PROPOSED CUT SLOPE ANGLE A (°)").grid(row=4, column=1, sticky="w", padx=15, pady=(5,0))
        self.ent_beta = ttk.Entry(self.soil_panel, width=30); self.ent_beta.insert(0, str(self.state["beta"])); self.ent_beta.grid(row=5, column=0, sticky="w", pady=(2, 10))
        self.ent_alpha = ttk.Entry(self.soil_panel, width=30); self.ent_alpha.insert(0, str(self.state["alpha"])); self.ent_alpha.grid(row=5, column=1, sticky="w", padx=15, pady=(2, 10))

        self.label_sm(self.soil_panel, "CUT HEIGHT H (M)").grid(row=6, column=0, sticky="w", pady=(5,0))
        self.label_sm(self.soil_panel, "GROUNDWATER Ru (0 to 1)").grid(row=6, column=1, sticky="w", padx=15, pady=(5,0))
        self.ent_H = ttk.Entry(self.soil_panel, width=30); self.ent_H.insert(0, str(self.state["H"])); self.ent_H.grid(row=7, column=0, sticky="w", pady=(2, 5))
        self.ent_ru = ttk.Entry(self.soil_panel, width=30); self.ent_ru.insert(0, str(self.state["ru"])); self.ent_ru.grid(row=7, column=1, sticky="w", padx=15, pady=(2, 5))
        tk.Label(self.soil_panel, text="(Hint: 0=Dry, 0.25=Partial, 0.5=Saturated)", font=("Segoe UI", 8, "italic"), bg="#F7F6F2", fg=C_TEXT_LIGHT).grid(row=8, column=1, sticky="w", padx=15, pady=(0, 10))
        
        self.label_sm(self.soil_panel, "NATURAL SLOPE HEIGHT H_nat (M)").grid(row=9, column=0, sticky="w", pady=(5,0))
        self.label_sm(self.soil_panel, "ANALYSIS ACCURACY LEVEL").grid(row=9, column=1, sticky="w", padx=15, pady=(5,0))
        self.ent_H_nat = ttk.Entry(self.soil_panel, width=30); self.ent_H_nat.insert(0, str(self.state["H_nat"])); self.ent_H_nat.grid(row=10, column=0, sticky="w", pady=(2, 5))
        self.cb_accuracy = ttk.Combobox(self.soil_panel, values=["1 - Average (Fast)", "2 - Fine (Slower)", "3 - Very Fine (Slowest)"], state="readonly", width=30); self.cb_accuracy.set("1 - Average (Fast)"); self.cb_accuracy.grid(row=10, column=1, sticky="w", padx=15, pady=(2, 5))

        # Rock Panel
        self.rock_panel = tk.Frame(self.card2, bg="#F7F6F2", highlightbackground=C_BORDER, highlightthickness=1, bd=0, padx=15, pady=15)
        self.label_sm(self.rock_panel, "ROCK TYPE GROUP").grid(row=0, column=0, sticky="w", pady=(5,0)); self.label_sm(self.rock_panel, "ROCK MASS CONDITION").grid(row=0, column=1, sticky="w", padx=15, pady=(5,0))
        self.cb_rock_g = ttk.Combobox(self.rock_panel, values=["G1 — Gneiss/Granite", "G2 — Quartzite/Diorite", "G3 — Sandstone/Basalt", "G4 — Siltstone/Phyllite/Schist"], state="readonly", width=30); self.cb_rock_g.set("G1 — Gneiss/Granite"); self.cb_rock_g.grid(row=1, column=0, sticky="w", pady=(2,10))
        self.cb_rock_m = ttk.Combobox(self.rock_panel, values=["blocky (3 joint sets)", "disturbed (many joints)", "disintegrated (heavily broken)"], state="readonly", width=30); self.cb_rock_m.set("blocky (3 joint sets)"); self.cb_rock_m.grid(row=1, column=1, sticky="w", padx=15, pady=(2,10))
        self.label_sm(self.rock_panel, "DISCONTINUITY ROUGHNESS").grid(row=2, column=0, sticky="w", pady=(5,0)); self.label_sm(self.rock_panel, "CUT HEIGHT H (M)").grid(row=2, column=1, sticky="w", padx=15, pady=(5,0))
        self.cb_rough = ttk.Combobox(self.rock_panel, values=["very_rough (fresh)", "rough (weathered)", "smooth (planar joints)", "slickensided (highly altered)"], state="readonly", width=30); self.cb_rough.set("rough (weathered)"); self.cb_rough.grid(row=3, column=0, sticky="w", pady=(2,10))
        self.ent_rH = ttk.Entry(self.rock_panel, width=30); self.ent_rH.insert(0, str(self.state["rH"])); self.ent_rH.grid(row=3, column=1, sticky="w", padx=15, pady=(2,10))
        self.label_sm(self.rock_panel, "BEDDING PLANE DIP (°)").grid(row=4, column=0, sticky="w", pady=(5,0)); self.label_sm(self.rock_panel, "BEDDING DIP DIRECTION").grid(row=4, column=1, sticky="w", padx=15, pady=(5,0))
        self.ent_bdip = ttk.Entry(self.rock_panel, width=30); self.ent_bdip.insert(0, str(self.state["bdip"])); self.ent_bdip.grid(row=5, column=0, sticky="w", pady=(2,10))
        self.cb_bdir = ttk.Combobox(self.rock_panel, values=["Dipping out of slope >30° (adverse)", "Dipping out of slope <30°", "Dipping into slope (favourable)"], state="readonly", width=30); self.cb_bdir.set("Dipping into slope (favourable)"); self.cb_bdir.grid(row=5, column=1, sticky="w", padx=15, pady=(2,10))

        ttk.Button(page, text="Continue to Stability Analysis →", style="Action.TButton", command=lambda: self.show_page(2)).pack(pady=15, padx=5, anchor="e")
        self.on_soil_updated(None)

    def build_page_2(self):
        page = tk.Frame(self.scrollable_frame, bg=C_BG); self.pages[2] = page
        card = self.create_card(page, "📉 Slope Stability Analysis (Fellenius)", "Slip circle computation utilizing Method of Slices")
        self.fig = Figure(figsize=(7, 3.5), dpi=100, facecolor="white"); self.ax = self.fig.add_subplot(111); self.ax.set_facecolor("white")
        self.chart_canvas = FigureCanvasTkAgg(self.fig, master=card); self.chart_canvas.get_tk_widget().pack(fill="x", expand=True, pady=10)
        
        ttk.Button(card, text="📉 Run Mathematical Assessment", style="Action.TButton", command=self.run_stability_analysis).pack(anchor="w", pady=5)
        self.results_frame = tk.Frame(card, bg="white"); self.results_frame.pack(fill="x", pady=10)
        self.lbl_slope_fs = tk.Label(self.results_frame, text="Slope FoS: --", font=("Segoe UI Bold", 13), bg="white"); self.lbl_slope_fs.pack(anchor="w", pady=2)
        
        self.dec_box = tk.Frame(card, bg="white", highlightbackground=C_BORDER, highlightthickness=1, bd=0); self.dec_box.pack(fill="x", pady=10)
        self.lbl_decision = tk.Label(self.dec_box, text="Please run stability assessment above.", wraplength=700, bg="white", justify="left"); self.lbl_decision.pack(fill="x", padx=10, pady=10)
        
        btn_row = tk.Frame(page, bg=C_BG); btn_row.pack(fill="x", pady=15, padx=5)
        ttk.Button(btn_row, text="← Back to Site Parameters", style="Nav.TButton", command=lambda: self.show_page(1)).pack(side="left")
        self.btn_next_p2 = ttk.Button(btn_row, text="Proceed to Retaining Wall Sizing →", style="Action.TButton", command=lambda: self.show_page(3)); self.btn_next_p2.pack(side="right")

    def build_page_3(self):
        page = tk.Frame(self.scrollable_frame, bg=C_BG); self.pages[3] = page
        self.bypass_wall_frame = tk.Frame(page, bg="white", highlightbackground=C_BORDER, highlightthickness=1, bd=0)
        tk.Label(self.bypass_wall_frame, text="✅ Structural retaining wall is not required.", font=("Segoe UI Semibold", 12), fg=C_SUCCESS, bg="white").pack(pady=20)
        ttk.Button(self.bypass_wall_frame, text="Proceed to Vegetative Bioengineering →", style="Action.TButton", command=lambda: self.show_page(4)).pack(pady=10)
        
        self.active_wall_frame = tk.Frame(page, bg=C_BG); self.active_wall_frame.pack(fill="both", expand=True)
        card = self.create_card(self.active_wall_frame, "🧱 Structural Retaining Wall Design (Rankine Theory)", "Base sliding & overturning check | IS:456 & IS:14458")
        inputs_grid = tk.Frame(card, bg="white"); inputs_grid.pack(fill="x", pady=5)
        self.label_sm(inputs_grid, "WALL HEIGHT HW (M)").grid(row=0, column=0, sticky="w", pady=5); self.ent_wH = ttk.Entry(inputs_grid, width=15); self.ent_wH.insert(0, str(self.state["wH"])); self.ent_wH.grid(row=0, column=1, sticky="w", padx=10, pady=5)
        self.label_sm(inputs_grid, "BASE WIDTH BB (M)").grid(row=0, column=2, sticky="w", pady=5); self.ent_wB = ttk.Entry(inputs_grid, width=15); self.ent_wB.insert(0, str(self.state["wB"])); self.ent_wB.grid(row=0, column=3, sticky="w", padx=10, pady=5)
        self.label_sm(inputs_grid, "TOP WIDTH A (M)").grid(row=1, column=0, sticky="w", pady=5); self.ent_wa = ttk.Entry(inputs_grid, width=15); self.ent_wa.insert(0, str(self.state["wa"])); self.ent_wa.grid(row=1, column=1, sticky="w", padx=10, pady=5)
        self.label_sm(inputs_grid, "FOUNDATION DEPTH DF (M)").grid(row=1, column=2, sticky="w", pady=5); self.ent_wDf = ttk.Entry(inputs_grid, width=15); self.ent_wDf.insert(0, str(self.state["wDf"])); self.ent_wDf.grid(row=1, column=3, sticky="w", padx=10, pady=5)
        self.label_sm(inputs_grid, "WALL TYPE").grid(row=2, column=0, sticky="w", pady=5); self.cb_wtype = ttk.Combobox(inputs_grid, values=["Gabion Mesh Wall (γ=20)", "Cement SMR 1:4 (γ=23)", "RCC Cantilever (γ=25)"], state="readonly", width=25); self.cb_wtype.set("Gabion Mesh Wall (γ=20)"); self.cb_wtype.grid(row=2, column=1, sticky="w", padx=10, pady=5)
        self.label_sm(inputs_grid, "FOUNDATION SBC (KPA)").grid(row=2, column=2, sticky="w", pady=5); self.ent_sbc = ttk.Entry(inputs_grid, width=15); self.ent_sbc.insert(0, str(self.state["wSbcVal"])); self.ent_sbc.grid(row=2, column=3, sticky="w", padx=10, pady=5)
        
        ttk.Button(card, text="🧱 Verify Wall Section Factors", style="Action.TButton", command=self.run_wall_design).pack(anchor="w", pady=10)
        self.table_frame = tk.Frame(card, bg="white"); self.table_frame.pack(fill="x", pady=5)
        for col_idx, h in enumerate(["Stability Criterion", "Required", "Calculated", "Status"]): self.label_sm(self.table_frame, h.upper()).grid(row=0, column=col_idx, padx=15, pady=5, sticky="w")
        
        self.lbl_overturning_calc = tk.Label(self.table_frame, text="--", bg="white"); self.lbl_overturning_calc.grid(row=1, column=2, padx=15, pady=3, sticky="w"); self.lbl_overturning_status = tk.Label(self.table_frame, text="--", font=("Segoe UI Bold", 9), bg="white"); self.lbl_overturning_status.grid(row=1, column=3, padx=15, pady=3, sticky="w"); tk.Label(self.table_frame, text="Factor of Safety (Overturning)", bg="white").grid(row=1, column=0, padx=15, pady=3, sticky="w"); tk.Label(self.table_frame, text="≥ 1.5", bg="white").grid(row=1, column=1, padx=15, pady=3, sticky="w")
        self.lbl_sliding_calc = tk.Label(self.table_frame, text="--", bg="white"); self.lbl_sliding_calc.grid(row=2, column=2, padx=15, pady=3, sticky="w"); self.lbl_sliding_status = tk.Label(self.table_frame, text="--", font=("Segoe UI Bold", 9), bg="white"); self.lbl_sliding_status.grid(row=2, column=3, padx=15, pady=3, sticky="w"); tk.Label(self.table_frame, text="Factor of Safety (Sliding)", bg="white").grid(row=2, column=0, padx=15, pady=3, sticky="w"); tk.Label(self.table_frame, text="≥ 1.5", bg="white").grid(row=2, column=1, padx=15, pady=3, sticky="w")
        self.lbl_ecc_calc = tk.Label(self.table_frame, text="--", bg="white"); self.lbl_ecc_calc.grid(row=3, column=2, padx=15, pady=3, sticky="w"); self.lbl_ecc_status = tk.Label(self.table_frame, text="--", font=("Segoe UI Bold", 9), bg="white"); self.lbl_ecc_status.grid(row=3, column=3, padx=15, pady=3, sticky="w"); tk.Label(self.table_frame, text="Base eccentricity (e)", bg="white").grid(row=3, column=0, padx=15, pady=3, sticky="w"); tk.Label(self.table_frame, text="≤ B/6", bg="white").grid(row=3, column=1, padx=15, pady=3, sticky="w")
        self.lbl_bearing_calc = tk.Label(self.table_frame, text="--", bg="white"); self.lbl_bearing_calc.grid(row=4, column=2, padx=15, pady=3, sticky="w"); self.lbl_bearing_status = tk.Label(self.table_frame, text="--", font=("Segoe UI Bold", 9), bg="white"); self.lbl_bearing_status.grid(row=4, column=3, padx=15, pady=3, sticky="w"); tk.Label(self.table_frame, text="Max base pressure", bg="white").grid(row=4, column=0, padx=15, pady=3, sticky="w"); tk.Label(self.table_frame, text="≤ SBC", bg="white").grid(row=4, column=1, padx=15, pady=3, sticky="w")

        self.wall_status_banner = tk.Frame(self.active_wall_frame, bg="white", highlightbackground=C_BORDER, highlightthickness=1, bd=0); self.wall_status_banner.pack(fill="x", pady=10)
        self.lbl_wall_status = tk.Label(self.wall_status_banner, text="Pending verification.", bg="white", font=("Segoe UI Semibold", 10)); self.lbl_wall_status.pack(pady=10)

        btn_row = tk.Frame(page, bg=C_BG); btn_row.pack(fill="x", pady=15, padx=5)
        ttk.Button(btn_row, text="← Back to Calculations", style="Nav.TButton", command=lambda: self.show_page(2)).pack(side="left")
        ttk.Button(btn_row, text="Continue to Bioengineering →", style="Action.TButton", command=lambda: self.show_page(4)).pack(side="right")

    def build_page_4(self):
        page = tk.Frame(self.scrollable_frame, bg=C_BG); self.pages[4] = page
        self.bio_card = self.create_card(page, "🌱 Bioengineering Design Spec Guidelines", "Based on Roadside Bioengineering Site Selector Rules")
        self.bio_content = tk.Frame(self.bio_card, bg="white"); self.bio_content.pack(fill="x")
        
        species_card = self.create_card(page, "🌸 Recommended Flora Species Selection", "Optimized target list for local bioengineering structures")
        self.table_species = tk.Frame(species_card, bg="white"); self.table_species.pack(fill="x")
        for col_idx, h in enumerate(["Local/Botanical Name", "Type", "Root Profile Type", "Slope Compatibility"]): self.label_sm(self.table_species, h.upper()).grid(row=0, column=col_idx, padx=15, pady=5, sticky="w")

        btn_row = tk.Frame(page, bg=C_BG); btn_row.pack(fill="x", pady=15, padx=5)
        ttk.Button(btn_row, text="← Back to Wall Sizing", style="Nav.TButton", command=lambda: self.show_page(3)).pack(side="left")
        ttk.Button(btn_row, text="Proceed to Drainage Design →", style="Action.TButton", command=lambda: self.show_page(5)).pack(side="right")

    def build_page_5(self):
        page = tk.Frame(self.scrollable_frame, bg=C_BG); self.pages[5] = page
        
        c1 = self.create_card(page, "🗺️ Catchment Area Delineation", "From topographic map / GIS / Google Earth (follow ridgelines)")
        tk.Label(c1, text="Field method (Google Earth): Enable Terrain layer → Shift+Scroll to 3D view → identify natural gullies → use Add Polygon tool tracing ridgelines.", font=("Segoe UI", 9), fg=C_INFO, bg=C_INFO_L, wraplength=950, justify="left", padx=10, pady=8).pack(fill="x", pady=(0, 15))
        f_grid1 = tk.Frame(c1, bg="white"); f_grid1.pack(fill="x")
        self.label_sm(f_grid1, "TOTAL CATCHMENT AREA A (KM²)").grid(row=0, column=0, sticky="w", pady=(0,5)); self.ent_dArea = ttk.Entry(f_grid1, width=25); self.ent_dArea.insert(0, str(self.state["dArea_km2"])); self.ent_dArea.grid(row=1, column=0, sticky="w", padx=(0, 20), pady=(0, 10))
        self.label_sm(f_grid1, "DRAIN TYPE").grid(row=0, column=1, sticky="w", pady=(0,5)); self.cb_drain_type = ttk.Combobox(f_grid1, values=["Both catch drain + side drain", "Road side drain", "Catch drain on the top of the slope"], state="readonly", width=35); self.cb_drain_type.set("Both catch drain + side drain"); self.cb_drain_type.grid(row=1, column=1, sticky="w", padx=(0, 20), pady=(0, 10))
        self.label_sm(f_grid1, "RETURN PERIOD").grid(row=0, column=2, sticky="w", pady=(0,5)); self.cb_return_pd = ttk.Combobox(f_grid1, values=["10-year (low hazard zone)", "25-year (critical landslide area)", "50-year (high risk infrastructure)"], state="readonly", width=35); self.cb_return_pd.set("50-year (high risk infrastructure)"); self.cb_return_pd.grid(row=1, column=2, sticky="w", pady=(0, 10))

        c2 = self.create_card(page, "🌴 Runoff Coefficient — Weighted C (Rational Formula)", "Enter % of catchment area for each land use type. Weighted C = Σ(Ci * Ai%) / 100")
        tk.Label(c2, text="Enter the percentage of catchment area (%) for each applicable land-use type. Total must equal 100%.", font=("Segoe UI", 9), fg=C_EARTH, bg=C_SAGE_LL, wraplength=950, justify="left", padx=10, pady=8).pack(fill="x", pady=(0, 15))
        table_frame = tk.Frame(c2, bg=C_BG2); table_frame.pack(fill="x")
        tk.Label(table_frame, text="Surface / Land Use Description", font=("Segoe UI", 9, "bold"), bg=C_EARTH_L, fg="white", anchor="w", padx=10).grid(row=0, column=0, sticky="ew", pady=1); tk.Label(table_frame, text="% of Area", font=("Segoe UI", 9, "bold"), bg=C_EARTH_L, fg="white", width=12).grid(row=0, column=1, sticky="ew", pady=1); tk.Label(table_frame, text="Contribution", font=("Segoe UI", 9, "bold"), bg=C_EARTH_L, fg="white", width=15).grid(row=0, column=2, sticky="ew", pady=1)
        table_frame.grid_columnconfigure(0, weight=1)

        self.runoff_vars = []; self.contrib_labels = []
        for idx, (desc, c_val) in enumerate(RUNOFF_TYPES, start=1):
            bg_color = "white" if idx % 2 == 1 else "#F9F9F9"
            tk.Label(table_frame, text=f"{desc} (C ~ {c_val:.2f})", bg=bg_color, anchor="w", padx=10).grid(row=idx, column=0, sticky="ew", pady=1)
            var = tk.StringVar(value="0"); var.trace_add("write", lambda *args: self.update_cw_totals()); self.runoff_vars.append((var, c_val))
            ent = tk.Entry(table_frame, textvariable=var, width=10, justify="center", bd=1, relief="solid"); ent.grid(row=idx, column=1, pady=3)
            lbl_contrib = tk.Label(table_frame, text="--", bg=bg_color); lbl_contrib.grid(row=idx, column=2, sticky="ew", pady=1); self.contrib_labels.append(lbl_contrib)

        tk.Label(table_frame, text="TOTAL", font=("Segoe UI", 9, "bold"), bg=C_SAGE_LL, fg=C_EARTH, anchor="w", padx=10).grid(row=9, column=0, sticky="ew", pady=2)
        self.lbl_total_pct = tk.Label(table_frame, text="0 %", font=("Segoe UI", 9, "bold"), bg=C_SAGE_LL, fg=C_EARTH); self.lbl_total_pct.grid(row=9, column=1, sticky="ew", pady=2)
        self.lbl_cw_final = tk.Label(table_frame, text="C_w = --", font=("Segoe UI", 9, "bold"), bg=C_SAGE_LL, fg=C_EARTH); self.lbl_cw_final.grid(row=9, column=2, sticky="ew", pady=2)

        c3 = self.create_card(page, "⏱️ Time of Concentration & Rainfall Intensity", "Kirpich (1940) — tc = 0.0195 * L^1.155 / H^0.385 (min) | For mountain catchments tc = 5 - 20 min")
        f_grid3 = tk.Frame(c3, bg="white"); f_grid3.pack(fill="x")
        self.label_sm(f_grid3, "FLOW PATH LENGTH L (M)").grid(row=0, column=0, sticky="w", pady=(0,5)); self.var_L = tk.StringVar(value=str(self.state["tc_L"])); self.var_L.trace_add("write", lambda *args: self.update_tc()); tk.Entry(f_grid3, textvariable=self.var_L, width=25).grid(row=1, column=0, sticky="w", padx=(0, 20), pady=(0, 10))
        self.label_sm(f_grid3, "HEIGHT DIFFERENCE H (M)").grid(row=0, column=1, sticky="w", pady=(0,5)); self.var_H = tk.StringVar(value=str(self.state["tc_H"])); self.var_H.trace_add("write", lambda *args: self.update_tc()); tk.Entry(f_grid3, textvariable=self.var_H, width=25).grid(row=1, column=1, sticky="w", padx=(0, 20), pady=(0, 10))

        mc_frame = tk.Frame(f_grid3, bg=C_BG2, padx=10, pady=10); mc_frame.grid(row=2, column=0, columnspan=2, sticky="ew", pady=(5, 10)); mc_frame.columnconfigure((0,1,2), weight=1)
        self.label_sm(mc_frame, "TC (KIRPICH)").grid(row=0, column=0); self.lbl_tc_raw = tk.Label(mc_frame, text="--", font=("Georgia", 14, "bold"), bg=C_BG2, fg=C_TEXT); self.lbl_tc_raw.grid(row=1, column=0); tk.Label(mc_frame, text="minutes", font=("Segoe UI", 8), bg=C_BG2, fg=C_TEXT_LIGHT).grid(row=2, column=0)
        self.label_sm(mc_frame, "TC (USED IN DESIGN)").grid(row=0, column=1); self.lbl_tc_used = tk.Label(mc_frame, text="--", font=("Georgia", 14, "bold"), bg=C_BG2, fg=C_SUCCESS); self.lbl_tc_used.grid(row=1, column=1); tk.Label(mc_frame, text="minutes", font=("Segoe UI", 8), bg=C_BG2, fg=C_TEXT_LIGHT).grid(row=2, column=1)
        self.label_sm(mc_frame, "RANGE (MOUNTAIN ROADS)").grid(row=0, column=2); tk.Label(mc_frame, text="5 - 20", font=("Georgia", 14, "bold"), bg=C_BG2, fg=C_TEXT).grid(row=1, column=2); tk.Label(mc_frame, text="minutes (typical)", font=("Segoe UI", 8), bg=C_BG2, fg=C_TEXT_LIGHT).grid(row=2, column=2)

        self.lbl_tc_advisory = tk.Label(f_grid3, text="", font=("Segoe UI", 9), fg=C_INFO, bg=C_INFO_L, wraplength=900, justify="left", padx=10, pady=5); self.lbl_tc_advisory.grid(row=3, column=0, columnspan=2, sticky="ew", pady=(0, 15)); self.lbl_tc_advisory.grid_remove()

        self.label_sm(f_grid3, "RAINFALL INTENSITY I (MM/HR)").grid(row=4, column=0, sticky="w", pady=(5,5)); self.ent_dI = ttk.Entry(f_grid3, width=25); self.ent_dI.insert(0, str(self.state["dI"])); self.ent_dI.grid(row=5, column=0, sticky="w", padx=(0, 20), pady=(0, 10))
        self.label_sm(f_grid3, "INTENSITY SOURCE").grid(row=4, column=1, sticky="w", pady=(5,5)); self.cb_isrc = ttk.Combobox(f_grid3, values=["Manually entered (DHM IDF data / field measurement)", "Typical Nepal mountain road — High rainfall zone (125 mm/hr)", "Typical Nepal mountain road — Medium rainfall zone (80 mm/hr)", "Typical Nepal mountain road — Low rainfall zone (50 mm/hr)"], state="readonly", width=55); self.cb_isrc.set("Manually entered (DHM IDF data / field measurement)"); self.cb_isrc.grid(row=5, column=1, sticky="w", pady=(0, 10)); self.cb_isrc.bind("<<ComboboxSelected>>", self.on_isrc_change)
        self.label_sm(f_grid3, "ANTECEDENT MOISTURE CONDITION (AMC)").grid(row=6, column=0, sticky="w", pady=(5,5)); self.cb_amc = ttk.Combobox(f_grid3, values=["AMC-II — Normal (average soil moisture, typical design)", "AMC-I — Dry (wilting point, low antecedent rainfall)", "AMC-III — Wet (field capacity, preceding 5-day rainfall >50 mm)"], state="readonly", width=45); self.cb_amc.set("AMC-II — Normal (average soil moisture, typical design)"); self.cb_amc.grid(row=7, column=0, sticky="w", padx=(0, 20), pady=(0, 10))
        self.label_sm(f_grid3, "CATCHMENT SLOPE CATEGORY").grid(row=6, column=1, sticky="w", pady=(5,5)); self.cb_catslope = ttk.Combobox(f_grid3, values=["Moderate (5–25%) — typical Himalayan road catchment", "Steep (>25%) — rapid runoff response, shorter tc", "Gentle (<5%) — slower runoff, longer tc"], state="readonly", width=45); self.cb_catslope.set("Moderate (5–25%) — typical Himalayan road catchment"); self.cb_catslope.grid(row=7, column=1, sticky="w", pady=(0, 10)); self.cb_catslope.bind("<<ComboboxSelected>>", lambda e: self.update_tc())

        c4 = self.create_card(page, "📐 Hydraulic Sizing (Manning's Flow Formula)", "V = (1/n) · R^(2/3) · S^(1/2) | Check limits against surface linings")
        f_grid4 = tk.Frame(c4, bg="white"); f_grid4.pack(fill="x")
        self.label_sm(f_grid4, "DRAIN LINING / SURFACE TYPE").grid(row=0, column=0, columnspan=2, sticky="w", pady=(0,5)); self.cb_lining = ttk.Combobox(f_grid4, values=list(LINING_DB.keys()), state="readonly", width=55); self.cb_lining.set("Cement stone masonry (1:4), random rubble"); self.cb_lining.grid(row=1, column=0, columnspan=2, sticky="w", padx=(0, 20), pady=(0, 10)); self.cb_lining.bind("<<ComboboxSelected>>", self.on_lining_sync)
        self.lbl_lining_info = tk.Label(f_grid4, text="Manning's n: -- | Max Velocity: -- m/s", font=("Segoe UI", 9, "italic"), fg=C_EARTH, bg=C_SAGE_LL, padx=10, pady=2); self.lbl_lining_info.grid(row=2, column=0, columnspan=2, sticky="w", pady=(0, 15))
        self.label_sm(f_grid4, "CHANNEL PROFILE").grid(row=3, column=0, sticky="w", pady=(0,5)); self.cb_profile = ttk.Combobox(f_grid4, values=["Trapezoidal", "Rectangular"], state="readonly", width=25); self.cb_profile.set("Trapezoidal"); self.cb_profile.grid(row=4, column=0, sticky="w", padx=(0, 20), pady=(0, 10)); self.cb_profile.bind("<<ComboboxSelected>>", self.on_profile_sync)
        self.label_sm(f_grid4, "BOTTOM WIDTH B (M)").grid(row=3, column=1, sticky="w", pady=(0,5)); self.ent_dtB = ttk.Entry(f_grid4, width=25); self.ent_dtB.insert(0, str(self.state["dtB"])); self.ent_dtB.grid(row=4, column=1, sticky="w", padx=(0, 20), pady=(0, 10))
        self.lbl_z = self.label_sm(f_grid4, "SIDE SLOPE (Z:1 H:V)"); self.lbl_z.grid(row=3, column=2, sticky="w", pady=(0,5)); self.ent_dtz = ttk.Entry(f_grid4, width=25); self.ent_dtz.insert(0, str(self.state["dtz"])); self.ent_dtz.grid(row=4, column=2, sticky="w", pady=(0, 10))
        self.label_sm(f_grid4, "TOTAL DEPTH D (M)").grid(row=5, column=0, sticky="w", pady=(5,5)); self.ent_dtD = ttk.Entry(f_grid4, width=25); self.ent_dtD.insert(0, str(self.state["dtD"])); self.ent_dtD.grid(row=6, column=0, sticky="w", padx=(0, 20), pady=(0, 10))
        self.label_sm(f_grid4, "LONGITUDINAL SLOPE S (%)").grid(row=5, column=1, sticky="w", pady=(5,5)); self.ent_dtS = ttk.Entry(f_grid4, width=25); self.ent_dtS.insert(0, str(self.state["dtS"])); self.ent_dtS.grid(row=6, column=1, sticky="w", padx=(0, 20), pady=(0, 10))

        ttk.Button(c4, text="📐 Calculate Drain Capacity & Safety", style="Action.TButton", command=self.run_drain_design).pack(anchor="w", pady=15)

        self.drain_results = tk.Frame(c4, bg="#F9F9F9", highlightbackground=C_BORDER, highlightthickness=1, bd=0, padx=15, pady=15); self.drain_results.pack(fill="x")
        self.lbl_qpeak = tk.Label(self.drain_results, text="Design Runoff (Q_peak): -- m³/s", font=("Segoe UI Semibold", 11), bg="#F9F9F9", fg=C_INFO); self.lbl_qpeak.pack(anchor="w", pady=2)
        self.lbl_qcap = tk.Label(self.drain_results, text="Channel Capacity (Q_cap): -- m³/s", font=("Segoe UI Semibold", 11), bg="#F9F9F9"); self.lbl_qcap.pack(anchor="w", pady=2)
        self.lbl_vflow = tk.Label(self.drain_results, text="Flow Velocity (V): -- m/s", font=("Segoe UI Semibold", 11), bg="#F9F9F9"); self.lbl_vflow.pack(anchor="w", pady=2)
        self.lbl_dsafety = tk.Label(self.drain_results, text="Awaiting calculation...", font=("Segoe UI", 10), bg="#F9F9F9", justify="left"); self.lbl_dsafety.pack(anchor="w", pady=(10,0))

        btn_row = tk.Frame(page, bg=C_BG); btn_row.pack(fill="x", pady=15, padx=5)
        ttk.Button(btn_row, text="← Back to Bioengineering", style="Nav.TButton", command=lambda: self.show_page(4)).pack(side="left")
        tk.Button(btn_row, text="🖨 Print / Save Full Report", bg=C_EARTH, fg="white", font=("Segoe UI Semibold", 10), bd=0, relief="flat", padx=15, pady=6, command=self.export_report).pack(side="right")

        self.on_lining_sync(None)

    # -----------------------------------------------------------------
    # GUI EVENT HANDLERS & CALCULATIONS
    # -----------------------------------------------------------------
    def toggle_material(self, material_type):
        self.state["mat"] = material_type
        if material_type == "soil":
            self.state["isRockMass"] = False
            self.btn_soil_mode.configure(bg=C_SAGE_LL, fg=C_EARTH); self.btn_rock_mode.configure(bg="white", fg=C_TEXT_LIGHT)
            self.soil_panel.pack(fill="x"); self.rock_panel.pack_forget()
        else:
            self.state["isRockMass"] = True
            self.btn_soil_mode.configure(bg="white", fg=C_TEXT_LIGHT); self.btn_rock_mode.configure(bg=C_SAGE_LL, fg=C_EARTH)
            self.rock_panel.pack(fill="x"); self.soil_panel.pack_forget()

    def sync_page_1_data(self):
        try:
            self.state["alt"] = float(self.ent_alt.get() or 0)
            self.state["aspect"] = self.cb_aspect.get()
            self.state["land"] = self.cb_land.get()
            self.state["moisture"] = self.cb_moist.get()
            self.state["rain"] = self.rain_var.get()
            if not self.state["isRockMass"]:
                self.state["beta"] = float(self.ent_beta.get()); self.state["alpha"] = float(self.ent_alpha.get())
                self.state["H"] = float(self.ent_H.get()); self.state["H_nat"] = float(self.ent_H_nat.get())
                self.state["ru"] = float(self.ent_ru.get())
                
                acc_val = self.cb_accuracy.get()
                if "1" in acc_val:
                    self.state["res"] = 30; self.state["slices"] = 30
                elif "2" in acc_val:
                    self.state["res"] = 50; self.state["slices"] = 50
                else:
                    self.state["res"] = 80; self.state["slices"] = 100
                
                if self.state["alpha"] <= self.state["beta"]: raise ValueError("Proposed cut slope must be steeper than the natural slope.")
                if self.state["ru"] < 0 or self.state["ru"] > 1: raise ValueError("Ru must be between 0 and 1.")
            else:
                self.state["rockGroup"] = self.cb_rock_g.get().split()[0]; self.state["rockMass"] = self.cb_rock_m.get().split()[0]; self.state["roughness"] = self.cb_rough.get().split()[0]
                self.state["rH"] = float(self.ent_rH.get()); self.state["bdip"] = float(self.ent_bdip.get()); self.state["bdir"] = self.cb_bdir.get()
            return True
        except ValueError as ve:
            messagebox.showerror("Validation Error", f"Invalid input format:\n{str(ve)}")
            return False

    def on_soil_updated(self, event):
        u_name = self.cb_soil.get()
        u_type = "CG"
        for k, v in SOIL_DB.items():
            if v["name"] == u_name: u_type = k; break
        str_cls = self.cb_str.get().split()[0]
        self.state["soilType"] = u_type; self.state["strClass"] = str_cls
        
        record = SOIL_DB[u_type]; params = record[str_cls]
        self.lbl_soil_hint.configure(text=f"{record['hint']}")
        self.lbl_c.configure(text=f"{params['c']}")
        self.lbl_phi.configure(text=f"{params['phi']}")
        self.lbl_gamma.configure(text=f"{params['g']}")

    def run_stability_analysis(self):
        if self.state["isRockMass"]:
            score = 100
            if self.state["rockGroup"] == "G4": score -= 20
            if "disturbed" in self.state["rockMass"]: score -= 15
            elif "disintegrated" in self.state["rockMass"]: score -= 30
            if "smooth" in self.state["roughness"]: score -= 10
            elif "slickensided" in self.state["roughness"]: score -= 20
            if "adverse" in self.state["bdir"].lower(): score -= 25

            self.lbl_slope_fs.configure(text=f"Rock Stability Index Score: {score}/100")
            if score >= 65: status, color, self.state["wallNeeded"] = "Stable Slope Face (Favourable Rock Joint Structure)", C_SUCCESS, False
            elif score >= 45: status, color, self.state["wallNeeded"] = "Marginal Ravelling Danger (Containment measures recommended)", C_AMBER, True
            else: status, color, self.state["wallNeeded"] = "Highly Unstable Block Structure (Structural Retaining required)", C_CLAY, True
            
            self.lbl_decision.configure(text=f"Assessment: {status}", fg=color)
            self.btn_next_p2.configure(text="Proceed to Structural Measures →" if self.state["wallNeeded"] else "Skip to Bioengineering Specs →")
            self.ax.clear(); self.ax.text(0.5, 0.5, f"Structural Rock Face Index: {score}\nFellenius method not applicable for Rock Mass.", ha='center', va='center', size=11, weight='bold', color=C_EARTH); self.ax.set_axis_off(); self.chart_canvas.draw()
            return

        soil = SOIL_DB[self.state["soilType"]][self.state["strClass"]]
        c = soil["c"]
        phi = soil["phi"]
        gamma = soil["g"]
        
        try:
            terrain_points = create_slope_geometry(self.state["H"], self.state["alpha"], self.state["H_nat"], self.state["beta"])
        except Exception as e:
            messagebox.showerror("Geometry Error", str(e))
            return

        self.ax.clear()
        self.ax.text(0.5, 0.5, "Calculating... Please Wait.", ha='center', va='center', size=12, color=C_TEXT_LIGHT)
        self.chart_canvas.draw()
        self.update()

        analyzer = SlopeStabilityAnalyzer(terrain_points, c, phi, gamma, self.state["ru"])
        fos, circle, data = analyzer.analyze(resolution=self.state["res"], n_slices=self.state["slices"], limit_entry_exit=True)

        self.state["fsCut"] = fos
        if fos == float('inf'):
             self.lbl_slope_fs.configure(text="Slope FoS: Error (No Valid Slip Circle)")
             status, color, self.state["wallNeeded"] = "Cannot compute realistic failure. Check inputs.", C_CLAY, True
        else:
             self.lbl_slope_fs.configure(text=f"Calculated Slope FoS: {fos:.3f}")
             if fos >= 1.3: status, color, self.state["wallNeeded"] = f"✅ STABLE CUT DESIGN. FS of {fos:.3f} satisfies standards.", C_SUCCESS, False
             elif fos >= 1.0: status, color, self.state["wallNeeded"] = f"⚠ MARGINAL PROFILE. Toe reinforcement wall required.", C_AMBER, True
             else: status, color, self.state["wallNeeded"] = f"❌ CRITICAL UNSAFE CUT. Structural retaining wall heavily required.", C_CLAY, True

        self.lbl_decision.configure(text=status, fg=color)
        self.btn_next_p2.configure(text="Proceed to Retaining Wall Sizing →" if self.state["wallNeeded"] else "Skip to Bioengineering Specs →")

        self.ax.clear()
        self.ax.set_axis_on()
        self.ax.set_title("Slip Circle Analysis (Fellenius)", fontsize=10, weight="bold", color=C_EARTH)
        self.ax.set_xlabel("Distance (m)", fontsize=8)
        self.ax.set_ylabel("Elevation (m)", fontsize=8)

        x_ground = np.linspace(analyzer.xs[0], analyzer.xs[-1], 300)
        y_ground = np.interp(x_ground, analyzer.xs, analyzer.ys)
        
        self.ax.plot(x_ground, y_ground, 'k-', linewidth=2)
        self.ax.fill_between(x_ground, y_ground, 0, color='#F0E68C', alpha=0.5)

        if circle:
            xc, yc, R = circle
            slices, x_entry, x_exit = data
            theta = np.linspace(np.arcsin(np.clip((x_entry-xc)/R, -1, 1)), 
                                np.arcsin(np.clip((x_exit-xc)/R, -1, 1)), 100)
            cx = xc + R * np.sin(theta)
            cy = yc - R * np.cos(theta)
            
            self.ax.fill_between(cx, cy, np.interp(cx, analyzer.xs, analyzer.ys), color='#00FF00', alpha=0.8)
            self.ax.plot(cx, cy, 'k-', linewidth=1.5)
            
            for s in slices:
                self.ax.plot([s['x'], s['x']], [float(np.interp(s['x'], analyzer.xs, analyzer.ys)), s['y']], 'k-', linewidth=0.5, alpha=0.5)
                
            self.ax.scatter([xc], [yc], color='red', marker='o', s=30, zorder=5)
            self.ax.plot([xc, cx[0]], [yc, cy[0]], 'k--', alpha=0.3)
            self.ax.plot([xc, cx[-1]], [yc, cy[-1]], 'k--', alpha=0.3)

        self.ax.axis('equal')
        self.fig.tight_layout()
        self.chart_canvas.draw()

    def run_wall_design(self):
        if not self.state["wallNeeded"]:
            self.active_wall_frame.pack_forget(); self.bypass_wall_frame.pack(fill="both", expand=True, padx=10, pady=10)
            return
        self.bypass_wall_frame.pack_forget(); self.active_wall_frame.pack(fill="both", expand=True)
        try:
            Hw = float(self.ent_wH.get()); B = float(self.ent_wB.get()); a = float(self.ent_wa.get()); Df = float(self.ent_wDf.get()); sbc = float(self.ent_sbc.get())
            w_choice = self.cb_wtype.get()
            if "Gabion" in w_choice: gamma_wall = 20.0; mu = 0.60
            elif "SMR" in w_choice: gamma_wall = 23.0; mu = 0.55
            else: gamma_wall = 25.0; mu = 0.50

            if not self.state["isRockMass"]: soil = SOIL_DB[self.state["soilType"]][self.state["strClass"]]; phi = soil["phi"]; gamma_s = soil["g"]
            else: phi = 32.0; gamma_s = 20.0

            fos_overturning, fos_sliding, ecc, q_max = calc_rankine_retaining_wall(Hw, B, a, Df, gamma_wall, mu, phi, gamma_s)

            self.lbl_overturning_calc.configure(text=f"{fos_overturning:.2f}"); self.lbl_overturning_status.configure(text="PASS", fg=C_SUCCESS) if fos_overturning >= 1.5 else self.lbl_overturning_status.configure(text="FAIL", fg=C_CLAY)
            self.lbl_sliding_calc.configure(text=f"{fos_sliding:.2f}"); self.lbl_sliding_status.configure(text="PASS", fg=C_SUCCESS) if fos_sliding >= 1.5 else self.lbl_sliding_status.configure(text="FAIL", fg=C_CLAY)
            self.lbl_ecc_calc.configure(text=f"{abs(ecc):.3f} m"); self.lbl_ecc_status.configure(text="PASS", fg=C_SUCCESS) if B > 0 and abs(ecc) <= (B / 6.0) else self.lbl_ecc_status.configure(text="FAIL", fg=C_CLAY)
            self.lbl_bearing_calc.configure(text=f"{q_max:.1f} kPa"); self.lbl_bearing_status.configure(text="PASS", fg=C_SUCCESS) if q_max <= sbc else self.lbl_bearing_status.configure(text="FAIL", fg=C_CLAY)

            if (fos_overturning >= 1.5) and (fos_sliding >= 1.5) and (abs(ecc) <= (B / 6.0)) and (q_max <= sbc): self.lbl_wall_status.configure(text="✅ SELECTED GEOMETRY SATISFIES SAFETY STANDARDS", fg=C_SUCCESS)
            else: self.lbl_wall_status.configure(text="❌ DESIGN UNSAFE. Re-scale section geometry thickness or width.", fg=C_CLAY)
        except Exception as e: messagebox.showerror("Numerical Error", f"Calculations error. Please review dimensions:\n{str(e)}")

    def build_bioengineering_view(self):
        for child in self.bio_content.winfo_children(): child.destroy()
        cut_angle = self.state["alpha"] if not self.state["isRockMass"] else self.state["sfa"]
        
        rec_techniques = []
        if self.state["isRockMass"]:
            if cut_angle > 60:
                rec_techniques.append(("Anchored Steel Mesh Netting", "Secures fractured joint blocks against direct structural falls"))
                rec_techniques.append(("Down-pipe subsoil drainage structures", "Relieves high cleft pressure spikes inside fissures"))
            else: rec_techniques.append(("Dry masonry rock-fill protection walls", "Fills major open scour cavities/caverns at base"))
        else:
            if cut_angle > 45:
                rec_techniques.append(("Live Brush Layering (Diagonal Rows)", "Reinforces surface layers with active root branches"))
                rec_techniques.append(("Woven Geotextile Jute Netting", "Restricts rain splash erosion on fresh, bare topsoil"))
            else: rec_techniques.append(("Palisades and Grass Slips (Vetiver Rows)", "Creates multi-layered root walls to trap silt runoff"))

        for title, desc in rec_techniques:
            card_sub = tk.Frame(self.bio_content, bg="white", highlightbackground=C_BORDER, highlightthickness=1, bd=0); card_sub.pack(fill="x", pady=4)
            tk.Label(card_sub, text=f"• {title}", font=("Segoe UI Semibold", 10), fg=C_EARTH, bg="white").pack(anchor="w", padx=10, pady=(5, 2))
            tk.Label(card_sub, text=desc, font=("Segoe UI", 9, "italic"), fg=C_TEXT_LIGHT, bg="white").pack(anchor="w", padx=25, pady=(0, 5))

        for child in self.table_species.winfo_children():
            if int(child.grid_info().get("row", 0)) > 0: child.destroy()
        altitude = self.state["alt"]
        if altitude > 1800: species_data = [("Utis (Alnus nepalensis)", "Fast Growing Tree", "Deep horizontal root network", "30° - 45° moist soil zones"), ("Phaledo (Erythrina arborescens)", "Staked Tree", "Broad lateral anchoring", "35° - 50° rugged slopes"), ("Babiyo (Eulaliopsis binata)", "Clump Grass", "High-tensile fibrous cluster", "40° - 60° dry stony soils")]
        else: species_data = [("Vetiver (Saccharum spontaneum)", "Deep Tap Grass", "Vertical root penetration (>2.5m)", "25° - 50° sandy washes"), ("Simali (Vitex negundo)", "Dense Shrub", "Thick surface erosion binding", "35° - 55° ravined slopes"), ("Bakaino (Melia azedarach)", "Medium Tree", "Stout anchoring peg roots", "30° - 45° stable clay soils")]

        for r_idx, (name, s_type, root, compatibility) in enumerate(species_data, start=1):
            tk.Label(self.table_species, text=name, font=("Segoe UI Semibold", 9), bg="white").grid(row=r_idx, column=0, padx=15, pady=4, sticky="w")
            tk.Label(self.table_species, text=s_type, bg="white").grid(row=r_idx, column=1, padx=15, pady=4, sticky="w")
            tk.Label(self.table_species, text=root, bg="white").grid(row=r_idx, column=2, padx=15, pady=4, sticky="w")
            tk.Label(self.table_species, text=compatibility, bg="white").grid(row=r_idx, column=3, padx=15, pady=4, sticky="w")

    def update_cw_totals(self):
        total_pct = 0.0; weighted_c = 0.0
        for i, (var, c_val) in enumerate(self.runoff_vars):
            try: val = float(var.get() or 0)
            except ValueError: val = 0.0
            total_pct += val; contrib = (val / 100.0) * c_val; weighted_c += contrib; self.contrib_labels[i].configure(text=f"{contrib:.3f}")
        
        self.current_cw = weighted_c
        self.lbl_total_pct.configure(text=f"{total_pct:.1f} %"); self.lbl_cw_final.configure(text=f"C_w = {weighted_c:.3f}")
        if total_pct > 100.1: self.lbl_total_pct.configure(fg=C_CLAY); self.lbl_cw_final.configure(fg=C_CLAY)
        else: self.lbl_total_pct.configure(fg=C_EARTH); self.lbl_cw_final.configure(fg=C_EARTH)

    def on_isrc_change(self, event):
        sel = self.cb_isrc.get()
        if "High" in sel: self.ent_dI.delete(0, tk.END); self.ent_dI.insert(0, "125")
        elif "Medium" in sel: self.ent_dI.delete(0, tk.END); self.ent_dI.insert(0, "80")
        elif "Low" in sel: self.ent_dI.delete(0, tk.END); self.ent_dI.insert(0, "50")

    def update_tc(self):
        try:
            L = float(self.var_L.get() or 0); H = float(self.var_H.get() or 0)
            tc_raw, tc_used = calc_kirpich_tc(L, H)
            
            self.state["tc_raw"] = tc_raw; self.state["tc_used"] = tc_used
            if tc_raw > 0:
                self.lbl_tc_raw.configure(text=f"{tc_raw:.2f}")
                self.lbl_tc_used.configure(text=f"{tc_used:.2f}" + (" (clamped)" if abs(tc_raw-tc_used)>0.01 else ""))
            else:
                self.lbl_tc_raw.configure(text="--"); self.lbl_tc_used.configure(text="--")

            cs = self.cb_catslope.get()
            if "Steep" in cs: csAdv = "Steep catchment — use tc toward lower end (5–10 min). Higher intensity applies."
            elif "Gentle" in cs: csAdv = "Gentle catchment — tc may approach upper limit (15–20 min). Lower intensity applies."
            else: csAdv = "Moderate catchment — Kirpich estimate appropriate."
            self.lbl_tc_advisory.configure(text=f"Slope category advisory: {csAdv}"); self.lbl_tc_advisory.grid() 
        except ValueError: self.lbl_tc_raw.configure(text="--"); self.lbl_tc_used.configure(text="--")

    def on_lining_sync(self, event):
        data = LINING_DB[self.cb_lining.get()]
        self.lbl_lining_info.configure(text=f"Design Manning's n: {data['n_disp']} (Avg: {data['n']})  |  Max Permissible Velocity: {data['v_disp']} m/s")

    def on_profile_sync(self, event):
        if self.cb_profile.get() == "Rectangular": self.lbl_z.grid_forget(); self.ent_dtz.grid_forget()
        else: self.lbl_z.grid(row=3, column=2, sticky="w", pady=(0,5)); self.ent_dtz.grid(row=4, column=2, sticky="w", pady=(0, 10))

    def run_drain_design(self):
        total_pct = sum([float(v.get() or 0) for v, c in self.runoff_vars])
        if total_pct > 100.1:
            messagebox.showerror("Validation Error", f"Total land use percentage exceeds 100% (currently at {total_pct:.1f}%). Please adjust variables to total 100%.")
            return
        if self.current_cw <= 0:
            messagebox.showerror("Validation Error", "Weighted Runoff Coefficient C_w is 0. Please assign land use percentages in the Rational Formula table.")
            return

        try:
            A_km2 = float(self.ent_dArea.get())
            if A_km2 <= 0: raise ValueError("Catchment area must be > 0.")
            I = float(self.ent_dI.get())

            Qpeak = calc_rational_discharge(self.current_cw, I, A_km2)
            self.lbl_qpeak.configure(text=f"Design Runoff (Q_peak): {Qpeak:.3f} m³/s")

            B = float(self.ent_dtB.get()); D = float(self.ent_dtD.get()); S_pct = float(self.ent_dtS.get())
            lining_data = LINING_DB[self.cb_lining.get()]
            n = lining_data["n"]; v_max = lining_data["v"]
            
            z = float(self.ent_dtz.get() or 0)
            
            Qcap, V = calc_manning_hydraulics(self.cb_profile.get(), B, D, z, S_pct, n)

            self.lbl_qcap.configure(text=f"Channel Capacity (Q_cap): {Qcap:.3f} m³/s")
            self.lbl_vflow.configure(text=f"Flow Velocity (V): {V:.2f} m/s")

            status = ""
            if Qcap >= Qpeak: status += "✅ CAPACITY ADEQUATE. Selected cross-section satisfies design runoff.\n"; color = C_SUCCESS
            else: status += "❌ UNDERSIZED CHANNEL. Flow capacity is lower than peak runoff.\n"; color = C_CLAY

            if V > v_max: status += f"⚠ WARNING: Flow velocity ({V:.2f} m/s) exceeds permissible limit for this lining ({v_max} m/s). Erosion/Scour risk. Add drops or change lining.\n"; color = C_CLAY if Qcap < Qpeak else C_AMBER
            elif V < 0.6: status += f"⚠ WARNING: Low self-cleansing velocity (<0.6 m/s). Siltation block risk. Steepen slope if possible.\n"; color = C_CLAY if Qcap < Qpeak else C_AMBER
            else: status += f"✅ Velocity is safe against erosion (V < {v_max} m/s) and siltation (V > 0.6 m/s)."

            self.lbl_dsafety.configure(text=status.strip(), fg=color)

        except ValueError as ve: messagebox.showerror("Validation Error", f"Missing or invalid dimensions:\n{str(ve)}")
        except Exception as e: messagebox.showerror("Numerical Error", f"Calculation error: {str(e)}")

    def export_report(self):
        filepath = filedialog.asksaveasfilename(defaultextension=".txt", filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")], title="Save Field Report")
        if not filepath: return
        report = []
        report.append("==================================================")
        report.append("  ROADSIDE SLOPE STABILIZATION & DRAINAGE REPORT  ")
        report.append(f"  Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report.append("==================================================\n")
        report.append("--- 1. SITE CHARACTERIZATION ---")
        report.append(f"Altitude: {self.state.get('alt', 'N/A')} m asl\nMaterial Type: {self.state.get('mat', 'soil').upper()}")
        if not self.state["isRockMass"]:
            report.append(f"Soil USCS: {self.state.get('soilType')} | Strength: {self.state.get('strClass')}\nNatural Angle: {self.state.get('beta')}° | Cut Angle: {self.state.get('alpha')}°\nCut Height: {self.state.get('H')} m | Groundwater ru: {self.state.get('ru')}")
        else: report.append(f"Rock Group: {self.state.get('rockGroup')} | Rock Mass: {self.state.get('rockMass')}")
        report.append("\n--- 2. STABILITY RESULTS ---")
        fs_str = f"{self.state.get('fsCut'):.3f}" if isinstance(self.state.get('fsCut'), float) else str(self.state.get('fsCut'))
        report.append(f"Calculated Factor of Safety (Fellenius): {fs_str}\nStructural Wall Required: {'YES' if self.state.get('wallNeeded') else 'NO'}")
        report.append("\n--- 3. DRAINAGE HYDROLOGY ---")
        try:
            A = float(self.ent_dArea.get()); I = float(self.ent_dI.get()); C = self.current_cw; Qp = (C * I * A * 100) / 360.0
            report.append(f"Catchment Area: {A} km2\nWeighted Runoff Coeff (Cw): {C:.3f}\nTime of Concentration (tc used): {self.state.get('tc_used', 0):.2f} min\nAMC Level: {self.cb_amc.get()}\nSlope Class: {self.cb_catslope.get()}\nDesign Peak Runoff (Qpeak): {Qp:.3f} m3/s")
        except: report.append("Hydrology data incomplete.")
        try:
            with open(filepath, "w", encoding="utf-8") as f: f.write("\n".join(report))
            messagebox.showinfo("Success", f"Report saved successfully to:\n{filepath}")
        except Exception as e: messagebox.showerror("Export Error", f"Failed to save file:\n{str(e)}")

if __name__ == "__main__":
    app = RoadsideStabilizationApp()
    app.mainloop()

## Updated Sn and its formula, need to fix the ru, and other terms also

In [2]:
import math
# =========================================================
# METHOD SELECTION LOGIC (Slope Stability Classification)
# =========================================================
# Case 1:
# If soil is very weak cohesive and highly frictional
# → planar / wedge failure is more likely
# Condition: c ≤ 2 kPa AND φ > 30°
# Method: WEDGE / INFINITE SLOPE
#
# Case 2:
# Transition soil (mixed behavior)
# → both circular and planar failure possible
# Condition: 2 < c ≤ 10 kPa AND φ > 30°
# Method: RUN BOTH (Taylor + Wedge) and take minimum FS
#
# Case 3:
# Normal c–φ soil (most common cut slopes)
# → circular failure dominates
# Method: TAYLOR STABILITY METHOD (or Bishop if available)
# =========================================================





# =========================================================
# 2D TAYLOR STABILITY TABLE (Sn values)
# rows = slope angle β
# cols = friction angle φ
# =========================================================

TAYLOR_TABLE = {
    90: {0:0.261, 5:0.239, 10:0.218, 15:0.199, 20:0.182, 25:0.166},
    75: {0:0.219, 5:0.195, 10:0.173, 15:0.152, 20:0.134, 25:0.117},
    60: {0:0.191, 5:0.162, 10:0.138, 15:0.116, 20:0.097, 25:0.079},
    45: {0:0.170, 5:0.136, 10:0.108, 15:0.083, 20:0.062, 25:0.044},
    30: {0:0.156, 5:0.110, 10:0.075, 15:0.046, 20:0.025, 25:0.009},
    15: {0:0.145, 5:0.068, 10:0.070, 15:0.023, 20:0.015, 25:0.010}
}

# =========================================================
# BILINEAR INTERPOLATION FOR Sn
# =========================================================

def bilinear_sn(beta, phi):

    betas = sorted(TAYLOR_TABLE.keys(), reverse=True)
    phis = sorted(next(iter(TAYLOR_TABLE.values())).keys())

    # Clamp beta
    beta = max(min(beta, betas[0]), betas[-1])

    # Clamp phi
    phi = max(min(phi, phis[-1]), phis[0])

    # Find beta bounds
    for i in range(len(betas)-1):
        b1, b2 = betas[i], betas[i+1]
        if b2 <= beta <= b1:
            break

    # Find phi bounds
    for j in range(len(phis)-1):
        p1, p2 = phis[j], phis[j+1]
        if p1 <= phi <= p2:
            break

    Q11 = TAYLOR_TABLE[b1][p1]
    Q12 = TAYLOR_TABLE[b1][p2]
    Q21 = TAYLOR_TABLE[b2][p1]
    Q22 = TAYLOR_TABLE[b2][p2]

    # interpolation factors
    t = (beta - b2) / (b1 - b2)
    u = (phi - p1) / (p2 - p1)

    sn = (
        Q11 * (1 - t) * (1 - u)
        + Q21 * t * (1 - u)
        + Q12 * (1 - t) * u
        + Q22 * t * u
    )

    return sn


# =========================================================
# TAYLOR METHOD (φ ≤ 30°)
# =========================================================

def fs_taylor(beta, phi, c, gamma, H):

    sn = bilinear_sn(beta, phi)

    fs = c / (sn * gamma * H)

    return fs, sn


# =========================================================
# INFINITE SLOPE METHOD (φ > 30°)
# =========================================================

def fs_infinite(beta, phi, c, gamma, H, ru):

    beta_r = math.radians(beta)
    phi_r = math.radians(phi)

    fs_cohesion = (
        c /
        (gamma * H * math.sin(beta_r) * math.cos(beta_r))
    )

    fs_friction = (
        math.tan(phi_r) /
        math.tan(beta_r)
    )

    fs = fs_cohesion + fs_friction

    # simplified pore-pressure correction
    fs *= (1 - ru)

    return fs


# =========================================================
# MAIN CONTROLLER
# =========================================================

def compute_fs(beta, phi, c, gamma, H, ru):

    if phi > 30:

        print("\nMethod Used: Infinite Slope Model")

        fs = fs_infinite(beta, phi, c, gamma, H, ru)

        return fs, None

    else:

        print("\nMethod Used: Taylor Stability Chart")

        fs, sn = fs_taylor(beta, phi, c, gamma, H)

        return fs, sn


# =========================================================
# USER INPUT
# =========================================================

beta = float(input("Enter slope angle β (deg): "))
phi = float(input("Enter friction angle φ (deg): "))
c = float(input("Enter cohesion c (kPa): "))
gamma = float(input("Enter unit weight γ (kN/m³): "))
H = float(input("Enter slope height H (m): "))
ru = float(input("Enter pore pressure ratio ru (0–1): "))

# =========================================================
# RUN
# =========================================================

fs, sn = compute_fs(beta, phi, c, gamma, H, ru)

# =========================================================
# RESULTS
# =========================================================

print("\n================================")
print("FINAL FACTOR OF SAFETY =", round(fs, 3))

if sn is not None:
    print("Taylor Stability Number (Sn) =", round(sn, 4))
else:
    print("Taylor Stability Number (Sn) = Not Used")

print("================================")

KeyboardInterrupt: Interrupted by user

## Culmann's and Taylor stability method (loop method) (manual verification is needed, both lookup chart and formula), only methodology is fixed here. This code has the culmann's method more accurate validated from sloope/W. Taylor method seems lacking large variation

In [1]:
import math

# =========================================================
# TAYLOR TABLE
# =========================================================

TAYLOR_TABLE = {
    90: {0:0.261, 5:0.239, 10:0.218, 15:0.199, 20:0.182, 25:0.166},
    75: {0:0.219, 5:0.195, 10:0.173, 15:0.152, 20:0.134, 25:0.117},
    60: {0:0.191, 5:0.162, 10:0.138, 15:0.116, 20:0.097, 25:0.079},
    45: {0:0.170, 5:0.136, 10:0.108, 15:0.083, 20:0.062, 25:0.044},
    30: {0:0.156, 5:0.110, 10:0.075, 15:0.046, 20:0.025, 25:0.009},
    15: {0:0.145, 5:0.068, 10:0.070, 15:0.023, 20:0.015, 25:0.010}
}

# =========================================================
# TAYLOR INTERPOLATION
# =========================================================

def bilinear_sn(beta, phi):

    betas = sorted(TAYLOR_TABLE.keys(), reverse=True)
    phis = sorted(next(iter(TAYLOR_TABLE.values())).keys())

    phi = max(min(phi, 25), 0)
    beta = max(min(beta, 90), 15)

    for i in range(len(betas)-1):
        b1, b2 = betas[i], betas[i+1]
        if b2 <= beta <= b1:
            break

    for j in range(len(phis)-1):
        p1, p2 = phis[j], phis[j+1]
        if p1 <= phi <= p2:
            break

    Q11 = TAYLOR_TABLE[b1][p1]
    Q12 = TAYLOR_TABLE[b1][p2]
    Q21 = TAYLOR_TABLE[b2][p1]
    Q22 = TAYLOR_TABLE[b2][p2]

    t = (beta - b2) / (b1 - b2)
    u = (phi - p1) / (p2 - p1)

    return (
        Q11*(1-t)*(1-u) +
        Q21*t*(1-u) +
        Q12*(1-t)*u +
        Q22*t*u
    )

# =========================================================
# TAYLOR FS (ONLY FOR CUT SLOPE)
# =========================================================

def fs_taylor(beta, phi, c, gamma, H):
    Sn = bilinear_sn(beta, phi)
    return c / (Sn * gamma * H), Sn

# =========================================================
# CULMANN WEDGE
# =========================================================

def wedge_fs(beta, alpha, c, phi, gamma, H):

    beta_r = math.radians(beta)
    alpha_r = math.radians(alpha)
    phi_r = math.radians(phi)

    x_crest = H / math.tan(beta_r)
    x_fail = H / math.tan(alpha_r)

    if x_fail <= x_crest:
        return None

    area = 0.5 * H * (x_fail - x_crest)
    W = gamma * area

    L = H / math.sin(alpha_r)

    driving = W * math.sin(alpha_r)
    resisting = c * L + W * math.cos(alpha_r) * math.tan(phi_r)

    if driving <= 0:
        return None

    return resisting / driving

# =========================================================
# CULMANN SEARCH
# =========================================================

def fs_culmann(beta, phi, c, gamma, H):

    best_fs = float("inf")
    best_alpha = None

    alpha_min = 5
    alpha_max = beta - 2

    if alpha_min >= alpha_max:
        return None, None

    beta_r = math.radians(beta)
    x_crest = H / math.tan(beta_r)

    max_backreach = 20 * H  # natural behavior (more freedom)

    for alpha in range(alpha_min, int(alpha_max) + 1, 2):

        alpha_r = math.radians(alpha)

        x_fail = H / math.tan(alpha_r)
        backreach = x_fail - x_crest

        if backreach <= 0:
            continue
        if backreach > max_backreach:
            continue

        fs = wedge_fs(beta, alpha, c, phi, gamma, H)

        if fs is None:
            continue

        if fs < best_fs:
            best_fs = fs
            best_alpha = alpha

    return best_fs, best_alpha

# =========================================================
# NATURAL SLOPE (CULMANN ONLY)
# =========================================================

def analyze_natural(beta, H, phi, c, gamma):

    print("\n==============================")
    print("NATURAL SLOPE (CULMANN ONLY)")
    print("==============================")

    fs_c, alpha = fs_culmann(beta, phi, c, gamma, H)

    if fs_c is None:
        print("Culmann not valid")
        return None, float("inf")

    print("Culmann FoS =", round(fs_c, 3))
    print("Critical α =", alpha)

    return fs_c, fs_c

# =========================================================
# CUT SLOPE (TAYLOR + CULMANN)
# =========================================================

def analyze_cut(beta, H, phi, c, gamma):

    print("\n==============================")
    print("CUT SLOPE (TAYLOR + CULMANN)")
    print("==============================")

    # Taylor
    fs_t, Sn = fs_taylor(beta, phi, c, gamma, H)
    print("Taylor FoS =", round(fs_t, 3))

    # Culmann
    fs_c, alpha = fs_culmann(beta, phi, c, gamma, H)

    if fs_c:
        print("Culmann FoS =", round(fs_c, 3))
        print("Critical α =", alpha)
        fs_min = min(fs_t, fs_c)
    else:
        print("Culmann not valid")
        fs_min = fs_t

    print("Governing FoS =", round(fs_min, 3))

    return fs_min

# =========================================================
# INPUT
# =========================================================

phi = float(input("φ (deg): "))
c = float(input("c (kPa): "))
gamma = float(input("γ (kN/m³): "))

print("\n--- NATURAL ---")
bn = float(input("βn: "))
Hn = float(input("Hn: "))

print("\n--- CUT ---")
bc = float(input("βc: "))
Hc = float(input("Hc: "))

# =========================================================
# RUN
# =========================================================

nat_fs, _ = analyze_natural(bn, Hn, phi, c, gamma)
cut_fs = analyze_cut(bc, Hc, phi, c, gamma)

# =========================================================
# FINAL COMPARISON
# =========================================================

print("\n==============================")
print("FINAL COMPARISON")
print("==============================")

print("Natural FoS (Culmann):", round(nat_fs, 3))
print("Cut FoS (Governing)   :", round(cut_fs, 3))

φ (deg):  4
c (kPa):  5
γ (kN/m³):  20



--- NATURAL ---


βn:  3
Hn:  6



--- CUT ---


βc:  9
Hc:  3



NATURAL SLOPE (CULMANN ONLY)
Culmann not valid

CUT SLOPE (TAYLOR + CULMANN)
Taylor FoS = 0.699
Culmann FoS = 5.088
Critical α = 5
Governing FoS = 0.699

FINAL COMPARISON


TypeError: type NoneType doesn't define __round__ method